# Feature Matrix + Correlation Analysis — All GPT Models

Loads every GPT feature CSV from `../data/gpt_feature_generation/`, parses nested JSON columns,
and produces per-model outputs (feature matrix, heatmaps, correlation network) in `../data/feature_matrix_gpt/<model>/`.

**Works on any machine** — all paths are relative to this notebook.


In [ ]:
import ast
import os
import warnings
from pathlib import Path

import matplotlib
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 150

# ── Relative paths (works for anyone who clones this repo) ────────────────────
DATA_DIR   = Path('../data').resolve()
GPT_DIR    = DATA_DIR / 'gpt_feature_generation'
RDKIT_PATH = DATA_DIR / 'maier_chemdesc_representation.tsv'
OUT_BASE   = DATA_DIR / 'feature_matrix_gpt'
OUT_BASE.mkdir(parents=True, exist_ok=True)

# The 2 most important GPT features (highlighted in all plots)
IMPORTANT_GPT = ['hetero_atom_count', 'hydrophobic_surface_fraction']

# ── Auto-discover all model CSVs ───────────────────────────────────────────────
gpt_csv_files = sorted(GPT_DIR.glob('gpt_features_*.csv'))
print(f'Found {len(gpt_csv_files)} GPT feature files:')
for f in gpt_csv_files:
    print(f'  {f.name}')


In [ ]:
NON_JSON_COLS = {'prestwick_ID', 'smiles_resolution'}

def safe_parse(val):
    if pd.isna(val):
        return {}
    try:
        return ast.literal_eval(str(val).strip())
    except Exception:
        return {}

def flatten_dict(d, prefix=''):
    out = {}
    for k, v in d.items():
        full_key = f'{prefix}{k}' if not prefix else f'{prefix}__{k}'
        if isinstance(v, dict):
            out.update(flatten_dict(v, full_key))
        elif isinstance(v, (list, tuple)):
            out[f'{full_key}__len'] = len(v)
            for i, elem in enumerate(v[:6]):
                if isinstance(elem, (int, float, bool)):
                    out[f'{full_key}__{i}'] = float(elem)
        elif isinstance(v, bool):
            out[full_key] = float(v)
        elif isinstance(v, (int, float)):
            out[full_key] = float(v)
    return out

def color_ticklabels(ax, important, axis='x'):
    labels = ax.get_xticklabels() if axis == 'x' else ax.get_yticklabels()
    for lbl in labels:
        t = lbl.get_text()
        if t in important:
            lbl.set_color('#A32D2D'); lbl.set_fontweight('bold')
        elif t.startswith('rdkit__'):
            lbl.set_color('#0F6E56')
        elif t.startswith('gpt__'):
            lbl.set_color('#993C1D')

def make_col_colors(cols, important):
    colors = []
    for c in cols:
        if c in important:           colors.append('#A32D2D')
        elif c.startswith('rdkit__'): colors.append('#1D9E75')
        else:                         colors.append('#D85A30')
    return pd.Series(colors, index=cols)


In [ ]:
def run_analysis(gpt_csv_path, rdkit_path, out_base, important_gpt):
    """Parse, flatten, correlate and plot one GPT model feature file."""
    model_tag = gpt_csv_path.stem
    out_dir   = out_base / model_tag
    out_dir.mkdir(parents=True, exist_ok=True)

    # 1. Load & parse nested JSON columns
    print(f'\n{"-"*60}')
    print(f'Model: {model_tag}')
    raw = pd.read_csv(gpt_csv_path)
    print(f'  Raw shape: {raw.shape}')

    records = []
    for _, row in raw.iterrows():
        flat = {'prestwick_ID': row.get('prestwick_ID')}
        for col in raw.columns:
            if col in NON_JSON_COLS:
                continue
            parsed = safe_parse(row[col])
            if isinstance(parsed, dict):
                flat.update(flatten_dict(parsed, prefix=col))
            elif isinstance(parsed, (int, float, bool)):
                flat[col] = float(parsed)
        records.append(flat)

    parsed_df = pd.DataFrame(records)
    print(f'  After JSON parse: {parsed_df.shape}')

    # 2. Clean: keep numeric, drop near-zero-variance
    id_col     = parsed_df[['prestwick_ID']].copy()
    feature_df = parsed_df.drop(columns=['prestwick_ID'], errors='ignore')
    feature_df = feature_df.apply(pd.to_numeric, errors='coerce')
    feature_df.dropna(axis=1, how='all', inplace=True)
    feature_df.fillna(0, inplace=True)
    feature_df = feature_df.loc[:, feature_df.std() > 1e-6]
    print(f'  Feature matrix: {feature_df.shape}')

    # 3. Save feature matrices
    feature_df.to_csv(out_dir / f'{model_tag}_features.csv', index=False)
    feature_with_id = pd.concat([id_col.reset_index(drop=True),
                                  feature_df.reset_index(drop=True)], axis=1)
    feature_with_id.to_csv(out_dir / f'{model_tag}_features_with_id.csv', index=False)
    print(f'  Saved feature matrix -> {out_dir}')

    # 4. Self-correlation heatmap (GPT only)
    corr_self = feature_df.corr(method='spearman')
    fig, ax = plt.subplots(figsize=(22, 18))
    sns.heatmap(corr_self, cmap='coolwarm', center=0,
                xticklabels=True, yticklabels=True, ax=ax,
                linewidths=0, rasterized=True)
    ax.set_title(f'Self-Correlation — {model_tag} GPT Features (Spearman)', fontsize=14)
    plt.tight_layout()
    plt.savefig(out_dir / f'{model_tag}_self_corr.png', dpi=200, bbox_inches='tight')
    plt.close()

    # 5. Load & align RDKit features
    rdkit_raw = pd.read_csv(rdkit_path, sep='\t')
    common = (set(feature_with_id['prestwick_ID'].dropna().astype(str))
              & set(rdkit_raw['prestwick_ID'].dropna().astype(str)))
    rdkit_raw['prestwick_ID']      = rdkit_raw['prestwick_ID'].astype(str)
    feature_with_id['prestwick_ID'] = feature_with_id['prestwick_ID'].astype(str)

    rdkit_filt = (rdkit_raw[rdkit_raw['prestwick_ID'].isin(common)]
                  .sort_values('prestwick_ID').reset_index(drop=True))
    gpt_filt   = (feature_with_id[feature_with_id['prestwick_ID'].isin(common)]
                  .sort_values('prestwick_ID').reset_index(drop=True))

    rdkit_feat = rdkit_filt.drop(columns=['prestwick_ID'], errors='ignore')
    rdkit_feat = rdkit_feat.apply(pd.to_numeric, errors='coerce').fillna(0)
    rdkit_feat = rdkit_feat.loc[:, rdkit_feat.std() > 1e-6]
    gpt_feat   = gpt_filt.drop(columns=['prestwick_ID'], errors='ignore')
    gpt_feat   = gpt_feat.apply(pd.to_numeric, errors='coerce').fillna(0)
    print(f'  RDKit: {rdkit_feat.shape}  |  GPT: {gpt_feat.shape}')

    # 6. Build combined matrix
    rdkit_feat.columns = ['rdkit__' + c for c in rdkit_feat.columns]
    gpt_feat.columns   = ['gpt__'   + c for c in gpt_feat.columns]
    combined  = pd.concat([rdkit_feat.reset_index(drop=True),
                            gpt_feat.reset_index(drop=True)], axis=1)
    n_rdkit   = rdkit_feat.shape[1]
    n_gpt     = gpt_feat.shape[1]
    important_prefixed = [f'gpt__{f}' for f in important_gpt
                          if f'gpt__{f}' in combined.columns]

    patches = [mpatches.Patch(color='#0F6E56', label='RDKit'),
               mpatches.Patch(color='#993C1D', label='GPT'),
               mpatches.Patch(color='#A32D2D', label='Important GPT')]

    # 7. Cross-correlation heatmap (Spearman)
    corr_sp = combined.corr(method='spearman')
    fig, ax = plt.subplots(figsize=(28, 24))
    sns.heatmap(corr_sp, cmap='coolwarm', center=0,
                xticklabels=True, yticklabels=True, ax=ax,
                linewidths=0, rasterized=True)
    for xy in [(n_rdkit, 0, n_gpt, n_rdkit), (0, n_rdkit, n_rdkit, n_gpt)]:
        ax.add_patch(mpatches.Rectangle(xy[:2], xy[2], xy[3],
                     fill=False, edgecolor='black', lw=1.5, linestyle='--'))
    ax.axvline(x=n_rdkit, color='black', lw=1.5)
    ax.axhline(y=n_rdkit, color='black', lw=1.5)
    color_ticklabels(ax, important_prefixed, 'x')
    color_ticklabels(ax, important_prefixed, 'y')
    ax.legend(handles=patches, loc='upper right', fontsize=9)
    ax.set_title(f'Cross-Correlation: RDKit | {model_tag} GPT (Spearman)', fontsize=14)
    plt.tight_layout()
    plt.savefig(out_dir / f'{model_tag}_cross_corr_spearman.png', dpi=200, bbox_inches='tight')
    plt.close()

    # 8. Cross-correlation heatmap (Kendall)
    corr_kd = combined.corr(method='kendall')
    fig, ax = plt.subplots(figsize=(28, 24))
    sns.heatmap(corr_kd, cmap='coolwarm', center=0,
                xticklabels=True, yticklabels=True, ax=ax,
                linewidths=0, rasterized=True)
    color_ticklabels(ax, important_prefixed, 'x')
    color_ticklabels(ax, important_prefixed, 'y')
    ax.axvline(x=n_rdkit, color='black', lw=1.5)
    ax.axhline(y=n_rdkit, color='black', lw=1.5)
    ax.legend(handles=patches, loc='upper right', fontsize=9)
    ax.set_title(f'Cross-Correlation: RDKit | {model_tag} GPT (Kendall)', fontsize=14)
    plt.tight_layout()
    plt.savefig(out_dir / f'{model_tag}_cross_corr_kendall.png', dpi=200, bbox_inches='tight')
    plt.close()

    # 9. Clustered heatmap
    col_colors = make_col_colors(combined.columns, important_prefixed)
    cg = sns.clustermap(corr_sp, cmap='coolwarm', center=0,
                        col_colors=col_colors, row_colors=col_colors,
                        xticklabels=True, yticklabels=True,
                        figsize=(30, 26), dendrogram_ratio=0.12,
                        colors_ratio=0.02, rasterized=True)
    color_ticklabels(cg.ax_heatmap, important_prefixed, 'x')
    color_ticklabels(cg.ax_heatmap, important_prefixed, 'y')
    cg.fig.suptitle(f'Clustered Cross-Correlation: RDKit vs {model_tag} GPT (Spearman)',
                    y=1.01, fontsize=14)
    cg.savefig(out_dir / f'{model_tag}_clustered.png', dpi=200, bbox_inches='tight')
    plt.close()

    # 10. Top RDKit correlates for important GPT features
    rdkit_cols = [c for c in corr_sp.columns if c.startswith('rdkit__')]
    for feat in important_prefixed:
        if feat not in corr_sp.index:
            continue
        row = corr_sp.loc[feat, rdkit_cols].abs()
        print(f'  {feat}  max|r|={row.max():.3f}')
        for k, v in row.nlargest(3).items():
            print(f'    {k}: {v:.3f}')

    # 11. MolWt scatter
    gpt_mw_cols   = [c for c in gpt_feat.columns  if 'weight' in c.lower() or 'molwt' in c.lower()]
    rdkit_mw_cols = [c for c in rdkit_feat.columns if 'molwt' in c.lower() or '__mw' in c.lower()]
    if gpt_mw_cols and rdkit_mw_cols:
        gpt_mw   = gpt_feat[gpt_mw_cols[0]].values
        rdkit_mw = rdkit_feat[rdkit_mw_cols[0]].values
        r_val, _ = spearmanr(gpt_mw, rdkit_mw)
        fig, ax  = plt.subplots(figsize=(7, 6))
        ax.scatter(rdkit_mw, gpt_mw, alpha=0.4, s=15, color='#185FA5')
        mn, mx = min(rdkit_mw.min(), gpt_mw.min()), max(rdkit_mw.max(), gpt_mw.max())
        ax.plot([mn, mx], [mn, mx], 'r--', lw=1.2, label='y=x')
        ax.set_xlabel('RDKit MolWt')
        ax.set_ylabel(f'GPT MolWt ({model_tag})')
        ax.set_title(f'MolWt: GPT vs RDKit  |  Spearman r={r_val:.3f}')
        ax.legend(fontsize=9)
        plt.tight_layout()
        plt.savefig(out_dir / f'{model_tag}_molwt.png', dpi=200, bbox_inches='tight')
        plt.close()

    # 12. Correlation network
    THRESHOLD = 0.6
    G = nx.Graph()
    for c in combined.columns:
        nt = ('important' if c in important_prefixed
              else ('rdkit' if c.startswith('rdkit__') else 'gpt'))
        G.add_node(c, node_type=nt)
    cols = list(corr_sp.columns)
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            if abs(corr_sp.iloc[i, j]) > THRESHOLD:
                G.add_edge(cols[i], cols[j], weight=float(abs(corr_sp.iloc[i, j])))
    pos = nx.spring_layout(G, seed=42, k=0.4)
    node_colors = ['#A32D2D' if G.nodes[n]['node_type'] == 'important'
                   else ('#1D9E75' if G.nodes[n]['node_type'] == 'rdkit' else '#D85A30')
                   for n in G.nodes()]
    node_sizes  = [300 if G.nodes[n]['node_type'] == 'important' else 60 for n in G.nodes()]
    edge_w = [G[u][v]['weight'] for u, v in G.edges()]
    fig, ax = plt.subplots(figsize=(18, 14))
    nx.draw_networkx_edges(G, pos, ax=ax,
                           width=[w * 1.5 for w in edge_w], alpha=0.25, edge_color='#888')
    nx.draw_networkx_nodes(G, pos, ax=ax,
                           node_color=node_colors, node_size=node_sizes, alpha=0.85)
    label_dict = {n: n.replace('gpt__', '') for n in important_prefixed if n in G.nodes()}
    nx.draw_networkx_labels(G, pos, labels=label_dict, ax=ax, font_size=9, font_color='white',
                            bbox=dict(boxstyle='round,pad=0.2', fc='#A32D2D', ec='none', alpha=0.85))
    legend_patches = [mpatches.Patch(color='#1D9E75', label='RDKit'),
                      mpatches.Patch(color='#D85A30', label='GPT'),
                      mpatches.Patch(color='#A32D2D', label='Important GPT')]
    ax.legend(handles=legend_patches, loc='upper left', fontsize=10)
    ax.set_title(
        f'Feature Correlation Network — {model_tag}  |r|>{THRESHOLD}'
        f'  (nodes:{G.number_of_nodes()}, edges:{G.number_of_edges()})',
        fontsize=13)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(out_dir / f'{model_tag}_network.png', dpi=200, bbox_inches='tight')
    plt.close()

    print(f'  Done -> {out_dir}')


In [ ]:
for gpt_csv in gpt_csv_files:
    run_analysis(gpt_csv, RDKIT_PATH, OUT_BASE, IMPORTANT_GPT)

print('\nAll models processed.')
